In [24]:
import pandas as pd
import numpy as np

In [25]:
data = [[1, 'Alice Johnson'], [2, 'Bob Smith'], [3, 'Carol Davis'], [4, 'David Wilson'], [5, 'Emma Brown']]
employees = pd.DataFrame({
    'employee_id': pd.Series(dtype='int'),
    'name': pd.Series(dtype='str')
})

employees = pd.concat([employees,pd.DataFrame(data,columns=employees.columns)])

data = [[1, 1, '2023-01-15', 2], [2, 1, '2023-04-15', 3], [3, 1, '2023-07-15', 4], [4, 1, '2023-10-15', 5], [5, 2, '2023-02-01', 3], [6, 2, '2023-05-01', 2], [7, 2, '2023-08-01', 4], [8, 2, '2023-11-01', 5], [9, 3, '2023-03-10', 1], [10, 3, '2023-06-10', 2], [11, 3, '2023-09-10', 3], [12, 3, '2023-12-10', 4], [13, 4, '2023-01-20', 4], [14, 4, '2023-04-20', 4], [15, 4, '2023-07-20', 4], [16, 5, '2023-02-15', 3], [17, 5, '2023-05-15', 2]]
performance_reviews = pd.DataFrame({
    'review_id': pd.Series(dtype='int'),
    'employee_id': pd.Series(dtype='int'),
    'review_date': pd.Series(dtype='datetime64[ns]'),
    'rating': pd.Series(dtype='float')  # Use float to accommodate decimal ratings
})
performance_reviews = pd.concat([performance_reviews,pd.DataFrame(data,columns=performance_reviews.columns)])

In [26]:
performance_reviews[performance_reviews['employee_id']==1]

,review_id,employee_id,review_date,rating
0,1,1,2023-01-15,2.0
1,2,1,2023-04-15,3.0
2,3,1,2023-07-15,4.0
3,4,1,2023-10-15,5.0


In [27]:
# Similar to using LAG(rating,1) #shift(1)
performance_reviews['prev_rating'] = performance_reviews.sort_values('review_date',ascending = True).groupby('employee_id')['rating'].shift(1)
# Similar to using LAG(rating,2) #shift(2)
performance_reviews['prev2_rating'] = performance_reviews.sort_values('review_date',ascending = True).groupby('employee_id')['rating'].shift(2) 

# QUALIFY ROW_NUMBER() OVER (PARTITION NY employee_id ORDER BY review_date ASC)= 1 ### tail(1)
latest_df = performance_reviews.sort_values('review_date', ascending= True).groupby('employee_id').tail(1)

latest_df['improvement_score'] = latest_df['rating'] - latest_df['prev2_rating']

latest_df[
    (latest_df['prev2_rating'].notna())
    &
    (
    (latest_df['prev2_rating'] < latest_df['prev_rating'])
    &
    (latest_df['prev_rating'] < latest_df['rating'])
    )
].merge(employees, on = "employee_id", how='left')[['employee_id','name','improvement_score']].\
    sort_values(['improvement_score','employee_id'],ascending=[False,True])


,employee_id,name,improvement_score
1,2,Bob Smith,3.0
0,1,Alice Johnson,2.0
2,3,Carol Davis,2.0
